# **SMS Spam Detection Pipeline using PySpark**

* **Overview**
  * This project aims to build an end-to-end machine learning pipeline for detecting spam SMS messages using PySpark. The system processes raw text data and transforms it into meaningful numerical features to train a classification model.
* **Objective**
  * The goal is to classify SMS messages into:
    - **Ham (0)** → legitimate messages
    - **Spam (1)** → unwanted or promotional messages
  
* **Workflow**
  1. **Data Ingestion** – Load dataset into PySpark DataFrame  
  2. **Data Cleaning & Preprocessing** – Handle missing values and encode labels  
  3. **Exploratory Data Analysis (EDA)** – Understand data distribution  
  4. **Feature Engineering** – Convert text into numerical features (TF-IDF)  
  5. **Model Training** – Train a Logistic Regression classifier  
  6. **Pipeline Construction** – Combine all steps into a reusable pipeline  
  7. **Model Evaluation** – Evaluate performance using classification metrics  
  8. **Model Persistence** – Save and reload the trained model  
  9. **Prediction** – Test model on new SMS messages  

* **Expected Outcome**
  * A robust and reusable pipeline capable of accurately detecting spam messages.

# **Phase one: Data Ingestion**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam (1).csv


In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SMS Spam Detection") \
    .getOrCreate()

In [ ]:
df = spark.read.csv("spam.csv", header=True, inferSchema=True)

In [ ]:
df.printSchema()

root
 |-- v1: string (nullable = true)
 |-- v2: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)



In [ ]:
df.show(5)

+----+--------------------+----+----+----+
|  v1|                  v2| _c2| _c3| _c4|
+----+--------------------+----+----+----+
| ham|Go until jurong p...|NULL|NULL|NULL|
| ham|Ok lar... Joking ...|NULL|NULL|NULL|
|spam|Free entry in 2 a...|NULL|NULL|NULL|
| ham|U dun say so earl...|NULL|NULL|NULL|
| ham|Nah I don't think...|NULL|NULL|NULL|
+----+--------------------+----+----+----+
only showing top 5 rows


# **Phase 2: Data Cleaning & Preprocessing**

In [ ]:
df = df.select("v1", "v2")

In [ ]:
df = df.withColumnRenamed("v1", "label") \
       .withColumnRenamed("v2", "message")

In [ ]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+-----+-------+
|label|message|
+-----+-------+
|    0|      1|
+-----+-------+



In [ ]:
df = df.dropna()

In [ ]:
df.show(5)

+-----+--------------------+
|label|             message|
+-----+--------------------+
|  ham|Go until jurong p...|
|  ham|Ok lar... Joking ...|
| spam|Free entry in 2 a...|
|  ham|U dun say so earl...|
|  ham|Nah I don't think...|
+-----+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import when

df = df.withColumn(
    "label",
    when(df.label == "spam", 1).otherwise(0)
)

In [ ]:
df.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|    1|  747|
|    0| 4826|
+-----+-----+



In [ ]:
df.printSchema()

root
 |-- label: integer (nullable = false)
 |-- message: string (nullable = true)



In [ ]:
df.select("label", "message").show(10, truncate=False)

+-----+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|message                                                                                                                                                         |
+-----+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0    |Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                                                 |
|0    |Ok lar... Joking wif u oni...                                                                                                                                   |
|1    |Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075o

# **Phase 3: Exploratory Data Analysis (EDA)**





 **In this phase, we performed a statistical analysis to distinguish between 'Ham' and 'Spam' messages. The key findings are:**

* **Message Length Correlation:**


  The statistical summary reveals a stark contrast between classes. Spam messages (Label 1) are significantly longer, averaging 138.4 characters, whereas Ham messages (Label 0) are shorter, averaging 71.07 characters.
* **Word Count:**

  Spam messages also exhibit higher word counts (24 words) compared to Ham (14 words). This suggests that spam often contains more descriptive, promotional, or urgent language.

* **Outlier Detection:**

  Through the describe() function, we identified extreme outliers, such as a message with 910 characters. Capturing these variations is crucial for the model to generalize well and avoid misclassification.

* **Significance:**

  The standard deviation and mean calculations confirm that message length is a highly discriminative feature. By feeding these metrics into our VectorAssembler (Phase 4), we significantly enhance the model's predictive power.

In [ ]:
from pyspark.sql.functions import length, avg, col, split, size, stddev

df = df.withColumn("msg_length", length(col("message")))
df = df.withColumn("word_count", size(split(col("message"), " ")))

print("Detailed Statistics for Message Length & Word Count:")
df.groupBy("label").agg(
    avg("msg_length").alias("avg_len"),
    avg("word_count").alias("avg_words"),
    stddev("msg_length").alias("stddev_len")
).show()

print("Checking for potential outliers (extremely long messages):")
df.select("label", "msg_length", "word_count").describe().show()

Detailed Statistics for Message Length & Word Count:
+-----+------------------+------------------+-----------------+
|label|           avg_len|         avg_words|       stddev_len|
+-----+------------------+------------------+-----------------+
|    1|138.45917001338688|23.892904953145916| 29.0285406539934|
|    0| 71.07065893079155|14.323456278491504|58.08781079329322|
+-----+------------------+------------------+-----------------+

Checking for potential outliers (extremely long messages):
+-------+-------------------+-----------------+------------------+
|summary|              label|       msg_length|        word_count|
+-------+-------------------+-----------------+------------------+
|  count|               5573|             5573|              5573|
|   mean|0.13403911717207967|80.10335546384353| 15.60613673066571|
| stddev|0.34072490905807057|59.68133321202668|11.421542269062721|
|    min|                  0|                2|                 1|
|    max|                  1|     

# **Phase 4: Feature Engineering**

**To prepare the raw text for the model, we designed a robust NLP pipeline:**
* **RegexTokenizer:**

  Unlike standard tokenizers, RegexTokenizer uses the \\W pattern to split text by any non-word character. This ensures that punctuation, symbols, and special characters—often found in spam messages (e.g., "FREE!!!")—are handled correctly without losing semantic meaning.
* **StopWordsRemover:**

  We eliminated common English "stop words" that do not contribute to classification, reducing noise and focusing on discriminative terms.
* **TF-IDF:**

*   HashingTF: Converts the tokens into a numerical vector of 1000 features based on term frequency.
*  IDF (Inverse Document Frequency): Rescales these features by penalizing common words and boosting the weight of rare, highly-predictive words (e.g., "claim", "urgent", "won").




* **VectorAssembler (Feature Fusion):**


  This is a critical optimization step. We merged the text-based TF-IDF features with our engineered numerical metadata (msg_length and word_count). By providing the model with a combined vector of content and structure, we ensure maximum predictive accuracy and better handling of the dataset's class imbalance.

In [ ]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, VectorAssembler

tokenizer = RegexTokenizer(inputCol="message", outputCol="words", pattern="\\W")

stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=1000)

idf = IDF(inputCol="raw_features", outputCol="tfidf_features")

assembler = VectorAssembler(
    inputCols=["tfidf_features", "msg_length", "word_count"],
    outputCol="features"
)

# **Phase 5: Model Training**

* In this phase, a Logistic Regression classifier was trained using the features generated from the previous feature engineering stage.
* Logistic Regression was chosen as a baseline classification algorithm due to its effectiveness in binary classification problems such as spam detection.
* The model learns patterns from the TF-IDF features along with additional engineered features such as message length and word count.
* The training process involved fitting the model on the training dataset, allowing it to learn the relationship between input features and the corresponding labels (spam or ham).

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

# **Phase 6: Pipeline Construction**

* A PySpark Pipeline was constructed to integrate feature extraction, and classification.
* The pipeline ensures reusability by encapsulating all transformations and the trained model into a single object. This allows the same sequence of operations to be applied consistently to training, testing, and new unseen data without manual intervention.
* The pipeline was trained once using the training dataset and subsequently used for prediction on both test data and new SMS inputs.

In [ ]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[
    tokenizer,
    stopwords_remover,
    hashing_tf,
    idf,
    assembler,
    lr
])

# **Phase 7: Model Evaluation**

In this phase, the performance of the trained model was evaluated using multiple classification metrics, including accuracy, precision, recall, and F1-score.

- **Accuracy** measures the overall correctness of the model's predictions.
- **Precision** indicates how many of the messages predicted as spam are actually spam. (avoid false alarms)
- **Recall** measures how many actual spam messages were correctly identified.(don’t miss spam)
- **F1-score** provides a balance between precision and recall, making it a more reliable metric for imbalanced datasets.

Additionally, predictions were compared against true labels using a confusion matrix to better understand the types of classification errors made by the model, such as false positives and false negatives.

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
model = pipeline.fit(train_df)

In [ ]:
predictions = model.transform(test_df)

In [ ]:
predictions.select("label", "prediction", "probability").show(5)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99999999999999...|
|    0|       0.0|           [1.0,0.0]|
|    0|       0.0|[0.99999999974441...|
|    0|       0.0|           [1.0,0.0]|
|    0|       0.0|           [1.0,0.0]|
+-----+----------+--------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

Accuracy: 0.9486461251167133


In [ ]:
precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions)

print("Precision:", precision)

Precision: 0.9479808702678885


In [ ]:
recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions)

print("Recall:", recall)

Recall: 0.9486461251167133


In [ ]:
f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print("F1-score:", f1)

F1-score: 0.948274300261114


In [ ]:
predictions.groupBy("label", "prediction").count().show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|   30|
|    0|       0.0|  898|
|    1|       1.0|  118|
|    0|       1.0|   25|
+-----+----------+-----+



* 0 → 0	correct ham (True Negatives)
* 1 → 1	correct spam (True Positives)
* 0 → 1	false positive (False Positives)
* 1 → 0	false negative (False Negatives)

# **Phase 9**

In [ ]:
model.save("sms_spam_pipeline_model_v2")

from pyspark.ml.pipeline import PipelineModel
loaded_model = PipelineModel.load("sms_spam_pipeline_model_v2")


loaded_predictions = loaded_model.transform(test_df)
loaded_predictions.select("label", "prediction", "probability").show(5)


+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99999999999999...|
|    0|       0.0|           [1.0,0.0]|
|    0|       0.0|[0.99999999974441...|
|    0|       0.0|           [1.0,0.0]|
|    0|       0.0|           [1.0,0.0]|
+-----+----------+--------------------+
only showing top 5 rows


In [ ]:
!pwd


/content


In [ ]:
!ls -l


total 992
drwxr-xr-x 1 root root   4096 Apr  2 13:31  sample_data
drwxr-xr-x 4 root root   4096 Apr 11 12:13  sms_spam_pipeline_model_v2
-rw-r--r-- 1 root root 503663 Apr 11 12:12 'spam (1).csv'
-rw-r--r-- 1 root root 503663 Apr 11 12:11  spam.csv


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import length, split, size, col


new_messages = [
    Row(message="Hey, are we still meeting tomorrow?"),
    Row(message="Don't forget to bring the documents."),
    Row(message="Congratulations! You won a free ticket. Call now!"),
    Row(message="URGENT! Your account has been compromised. Click here to secure it.")
]

new_data = spark.createDataFrame(new_messages)


new_data = new_data.withColumn("msg_length", length(col("message")))
new_data = new_data.withColumn("word_count", size(split(col("message"), " ")))


new_predictions = loaded_model.transform(new_data)


new_predictions.select("message", "prediction", "probability").show(truncate=False)


+-------------------------------------------------------------------+----------+------------------------------------------+
|message                                                            |prediction|probability                               |
+-------------------------------------------------------------------+----------+------------------------------------------+
|Hey, are we still meeting tomorrow?                                |0.0       |[1.0,0.0]                                 |
|Don't forget to bring the documents.                               |0.0       |[0.9999999999999996,4.440892098500626E-16]|
|Congratulations! You won a free ticket. Call now!                  |1.0       |[1.2763499656948188E-6,0.9999987236500343]|
|URGENT! Your account has been compromised. Click here to secure it.|0.0       |[0.9999967794561408,3.220543859150382E-6] |
+-------------------------------------------------------------------+----------+------------------------------------------+

